In [1]:
%%capture
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
!git clone https://github.com/AI4Bharat/NeMo.git 
%cd NeMo 
!git checkout nemo-v2 
!bash reinstall.sh


Cloning into 'NeMo'...
remote: Enumerating objects: 79449, done.
remote: Counting objects: 100% (369/369), done.
remote: Compressing objects: 100% (159/159), done.
remote: Total 79449 (delta 277), reused 245 (delta 210), pack-reused 79080 (from 1)
Receiving objects: 100% (79449/79449), 140.99 MiB | 36.75 MiB/s, done.
Resolving deltas: 100% (57694/57694), done.
/kaggle/working/NeMo
Already on 'nemo-v2'
Your branch is up to date with 'origin/nemo-v2'.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 31.2 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
Uninstalling stuff
Installing nemo
Obtaining file:///kaggle/working/NeMo
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
  Pr

In [3]:
%%capture
!pip install torch
!pip install transformers
!pip install torchaudio
!pip install librosa


In [4]:
import os
print(os.path.exists("/kaggle/input/bn-to-en/bengali/dev/wav/-Mpqqv-wIqI.bn_24.wav"))


True


In [5]:
import torch
import nemo.collections.asr as nemo_asr
import torchaudio
import torchaudio.transforms as transforms
import uuid

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load IndicConformer model
model = nemo_asr.models.ASRModel.from_pretrained("ai4bharat/indicconformer_stt_bn_hybrid_rnnt_large")
model.freeze()  # Set model to inference mode
model = model.to(device)  # Move model to GPU if available
model.cur_decoder = "rnnt"  # Set decoder type

# Function to preprocess audio
def preprocess_audio(waveform, sample_rate):
    target_sample_rate = 16000

    # Resample only if necessary
    if sample_rate != target_sample_rate:
        waveform = torchaudio.transforms.Resample(orig_freq=sample_rate, new_freq=target_sample_rate)(waveform)

    # Normalize volume
    if waveform.abs().max() > 0:
        waveform = waveform / waveform.abs().max()
    
    # Apply a slight gain boost
    volume_transform = transforms.Vol(gain=5, gain_type="amplitude")
    waveform = volume_transform(waveform)

    # Prevent saving empty waveforms
    if waveform.numel() == 0:
        raise ValueError("Audio file became empty after preprocessing!")

    return waveform


# Function to transcribe speech
def transcribe_audio(file_path):
    # Load and preprocess audio
    waveform, sample_rate = torchaudio.load(file_path)
    waveform = preprocess_audio(waveform, sample_rate)

    # Debug: Check waveform shape
    print("\nProcessed Audio Shape:", waveform.shape)

    # Save processed audio to a temporary file
    temp_file = f"processed_{uuid.uuid4().hex}.wav"
    torchaudio.save(temp_file, waveform, 16000)

    # Transcribe audio
    transcription = model.transcribe([temp_file], language_id="bn")

    # Debug: Print the raw output
    print("\nRaw Transcription Output:", transcription)
    print("Type of transcription:", type(transcription))

    # Flatten and clean transcription output
    if isinstance(transcription, tuple):
        transcription = list(transcription)

    flat_transcription = []
    for sublist in transcription:
        if isinstance(sublist, list):
            flat_transcription.extend(sublist)
        else:
            flat_transcription.append(sublist)

    # Remove empty strings
    flat_transcription = [str(item) for item in flat_transcription if item]

    return " ".join(flat_transcription) if flat_transcription else "No transcription available"


# Example usage
audio_file = "/kaggle/input/bn-to-en/bengali/dev/wav/-Mpqqv-wIqI.bn_24.wav"
transcription = transcribe_audio(audio_file)
print("\nFinal Transcription:", transcription)


[NeMo W 2025-03-10 19:09:38 nemo_logging:349] /usr/local/lib/python3.10/dist-packages/megatron/core/tensor_parallel/layers.py:220: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
      def forward(ctx, input, weight, bias, gradient_accumulation_fusion,
    
[NeMo W 2025-03-10 19:09:38 nemo_logging:349] /usr/local/lib/python3.10/dist-packages/megatron/core/tensor_parallel/layers.py:250: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
      def backward(ctx, grad_output):
    


Using device: cuda


(…)cconformer_stt_bn_hybrid_rnnt_large.nemo:   0%|          | 0.00/523M [00:00<?, ?B/s]

[NeMo I 2025-03-10 19:09:55 mixins:198] _setup_tokenizer: detected an aggregate tokenizer
[NeMo I 2025-03-10 19:09:55 mixins:335] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2025-03-10 19:09:55 mixins:335] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2025-03-10 19:09:55 mixins:335] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2025-03-10 19:09:55 mixins:335] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2025-03-10 19:09:55 mixins:335] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2025-03-10 19:09:55 mixins:335] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2025-03-10 19:09:55 mixins:335] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2025-03-10 19:09:55 mixins:335] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2025-03-10 19:09:55 mixins:335] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[

[NeMo W 2025-03-10 19:10:03 modelPT:165] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath:
    - /nlsasfs/home/ai4bharat/ai4bharat-pr/speechteam/indicasr_v3/manifests/nemo/vistaar_v3/train/train_bengali.json
    sample_rate: 16000
    batch_size: 8
    shuffle: false
    num_workers: 16
    pin_memory: true
    max_duration: 30.0
    min_duration: 0.2
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: synced_randomized
    bucketing_batch_size: null
    is_concat: true
    concat_sampling_technique: temperature
    concat_sampling_temperature: 1.5
    return_language_id: true
    
[NeMo W 2025-03-10 19:10:03 modelPT:172] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configurati

...

[NeMo I 2025-03-10 19:10:06 rnnt:1667] Vocab size for each language: 256
[NeMo I 2025-03-10 19:10:06 rnnt_models:220] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2025-03-10 19:10:06 rnnt_models:220] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2025-03-10 19:10:08 rnnt_models:220] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2025-03-10 19:10:08 hybrid_rnnt_ctc_bpe_models:105] Creating masks for multi-softmax layer.
[NeMo I 2025-03-10 19:10:08 rnnt_models:220] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}


[NeMo W 2025-03-10 19:10:08 nemo_logging:349] /kaggle/working/NeMo/nemo/core/connectors/save_restore_connector.py:585: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
      retu

[NeMo I 2025-03-10 19:10:09 save_restore_connector:263] Model EncDecHybridRNNTCTCBPEModel was successfully restored from /root/.cache/huggingface/hub/models--ai4bharat--indicconformer_stt_bn_hybrid_rnnt_large/snapshots/15a1cd06245262b914f27ed445a604b7b278d187/indicconformer_stt_bn_hybrid_rnnt_large.nemo.

Processed Audio Shape: torch.Size([1, 340480])


Transcribing:   0%|          | 0/1 [00:00<?, ?it/s][NeMo W 2025-03-10 19:10:09 nemo_logging:349] /kaggle/working/NeMo/nemo/collections/asr/parts/preprocessing/features.py:417: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
      with torch.cuda.amp.autocast(enabled=False):
    
Transcribing: 100%|██████████| 1/1 [00:02<00:00,  2.01s/it]


Raw Transcription Output: ([' সমীকরণটি বলে যে সমান যা এখানে সম্ভব নয় এর অর্থ হল আমরা এই সিস্টেমটি সমাধান করতে পারব না ল্যামডাউন এবং ল্যামডা টু এর জন্য এখানে কোন সমাধান নেই কারণ আমাদের সমীকরণগুলোই ইনকনসিস্ট্যান্ট'], [' সমীকরণটি বলে যে সমান যা এখানে সম্ভব নয় এর অর্থ হল আমরা এই সিস্টেমটি সমাধান করতে পারব না ল্যামডাউন এবং ল্যামডা টু এর জন্য এখানে কোন সমাধান নেই কারণ আমাদের সমীকরণগুলোই ইনকনসিস্ট্যান্ট'])
Type of transcription: <class 'tuple'>

Final Transcription:  সমীকরণটি বলে যে সমান যা এখানে সম্ভব নয় এর অর্থ হল আমরা এই সিস্টেমটি সমাধান করতে পারব না ল্যামডাউন এবং ল্যামডা টু এর জন্য এখানে কোন সমাধান নেই কারণ আমাদের সমীকরণগুলোই ইনকনসিস্ট্যান্ট  সমীকরণটি বলে যে সমান যা এখানে সম্ভব নয় এর অর্থ হল আমরা এই সিস্টেমটি সমাধান করতে পারব না ল্যামডাউন এবং ল্যামডা টু এর জন্য এখানে কোন সমাধান নেই কারণ আমাদের সমীকরণগুলোই ইনকনসিস্ট্যান্ট


In [6]:
print(model.cfg.decoding)


{'strategy': 'greedy_batch', 'greedy': {'max_symbols': 10}, 'beam': {'beam_size': 2, 'return_best_hypothesis': False, 'score_norm': True, 'tsd_max_sym_exp': 50, 'alsd_max_target_len': 2.0}}


In [7]:
model.cfg.decoding.strategy = "beam"
model.cfg.decoding.beam.beam_size = 5  # Change beam size as needed
print(model.cfg.decoding.strategy)


beam


In [8]:

# Example usage
audio_file = "/kaggle/input/bn-to-en/bengali/dev/wav/-Mpqqv-wIqI.bn_24.wav"
transcription = transcribe_audio(audio_file)
print("\nFinal Transcription:", transcription)
print(model.cfg.decoding.strategy)


Processed Audio Shape: torch.Size([1, 340480])


Transcribing: 100%|██████████| 1/1 [00:00<00:00,  1.47it/s]


Raw Transcription Output: ([' সমীকরণটি বলে যে সমান যা এখানে সম্ভব নয় এর অর্থ হল আমরা এই সিস্টেমটি সমাধান করতে পারব না ল্যামডাউন এবং ল্যামডা টু এর জন্য এখানে কোন সমাধান নেই কারণ আমাদের সমীকরণগুলোই ইনকনসিস্ট্যান্ট'], [' সমীকরণটি বলে যে সমান যা এখানে সম্ভব নয় এর অর্থ হল আমরা এই সিস্টেমটি সমাধান করতে পারব না ল্যামডাউন এবং ল্যামডা টু এর জন্য এখানে কোন সমাধান নেই কারণ আমাদের সমীকরণগুলোই ইনকনসিস্ট্যান্ট'])
Type of transcription: <class 'tuple'>

Final Transcription:  সমীকরণটি বলে যে সমান যা এখানে সম্ভব নয় এর অর্থ হল আমরা এই সিস্টেমটি সমাধান করতে পারব না ল্যামডাউন এবং ল্যামডা টু এর জন্য এখানে কোন সমাধান নেই কারণ আমাদের সমীকরণগুলোই ইনকনসিস্ট্যান্ট  সমীকরণটি বলে যে সমান যা এখানে সম্ভব নয় এর অর্থ হল আমরা এই সিস্টেমটি সমাধান করতে পারব না ল্যামডাউন এবং ল্যামডা টু এর জন্য এখানে কোন সমাধান নেই কারণ আমাদের সমীকরণগুলোই ইনকনসিস্ট্যান্ট
beam


In [9]:
import os
import torch
import pandas as pd
import time
import nemo.collections.asr as nemo_asr
import torchaudio
import torchaudio.transforms as transforms
import uuid

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load IndicConformer model
model = nemo_asr.models.ASRModel.from_pretrained("ai4bharat/indicconformer_stt_bn_hybrid_rnnt_large")
model.freeze()  # Set model to inference mode
model = model.to(device)  # Move model to GPU if available
model.cur_decoder = "rnnt"  # Set decoder type

model.cfg.decoding.strategy = "beam"
model.cfg.decoding.beam.beam_size = 10  # Change beam size as needed
print(model.cfg.decoding.strategy)

# Paths
audio_dir = "/kaggle/input/bn-to-en/bengali/dev/wav/"
output_csv = "bengali_transcriptions_dev11.csv"
log_file = "transcription_log.txt"

# List all audio files
wav_files = [f for f in os.listdir(audio_dir) if f.endswith(".wav")]

# Load existing transcriptions to avoid reprocessing
if os.path.exists(output_csv):
    existing_df = pd.read_csv(output_csv)
    existing_df.dropna(inplace=True)  # Remove any corrupted rows
    existing_df.to_csv(output_csv, index=False, encoding="utf-8")
    processed_files = set(existing_df["filename"].tolist())
else:
    processed_files = set()

# Filter unprocessed files
wav_files = [f for f in wav_files if f not in processed_files]

# Preprocessing function
def preprocess_audio(file_path):
    """Load and preprocess audio file."""
    waveform, sample_rate = torchaudio.load(file_path)

    # Resample if needed
    if sample_rate != 16000:
        resampler = transforms.Resample(orig_freq=sample_rate, new_freq=16000)
        waveform = resampler(waveform)

    # Normalize volume
    if waveform.abs().max() > 0:
        waveform = waveform / waveform.abs().max()

    # Apply noise reduction (Low-pass filter)
    waveform = torchaudio.functional.lowpass_biquad(waveform, sample_rate=16000, cutoff_freq=5000)

    # Apply a slight gain boost
    volume_transform = transforms.Vol(gain=5, gain_type="amplitude")
    waveform = volume_transform(waveform)

    # Prevent saving empty waveforms
    if waveform.numel() == 0:
        raise ValueError(f"Audio file {file_path} became empty after preprocessing!")

    # Save processed audio
    temp_file = f"processed_{uuid.uuid4().hex}.wav"
    torchaudio.save(temp_file, waveform, 16000)
    return temp_file  # Return temp file path


def transcribe_audio(file_path):
    """Preprocess and transcribe Bengali audio using IndicConformer."""
    processed_path = preprocess_audio(file_path)  # Apply preprocessing
    transcription = model.transcribe([processed_path], batch_size=1, language_id="bn")[0]
    os.remove(processed_path)  # Delete temp file after transcription
    return transcription


# Track start time
start_time = time.time()
time_limit = 11 * 60 * 60  # 11 hours in seconds
transcriptions = []

for i, file in enumerate(wav_files):
    file_path = os.path.join(audio_dir, file)

    # Transcribe audio
    print(f"Transcribing ({i+1}/{len(wav_files)}): {file}")
    try:
        text = transcribe_audio(file_path)
        transcriptions.append([file, text])
    except Exception as e:
        with open(log_file, "a") as log:
            log.write(f"Error processing {file}: {str(e)}\n")
        continue

    # Periodically save to CSV
    if (i + 1) % 10 == 0:  # Save every 10 files
        temp_df = pd.DataFrame(transcriptions, columns=["filename", "transcription"])
        if os.path.exists(output_csv):
            temp_df.to_csv(output_csv, mode="a", header=False, index=False, encoding="utf-8")
        else:
            temp_df.to_csv(output_csv, index=False, encoding="utf-8")
        transcriptions = []  # Clear memory

    # Check if time limit is exceeded
    if time.time() - start_time > time_limit:
        print("Time limit reached. Saving progress and stopping.")
        break

# Final save
if transcriptions:
    temp_df = pd.DataFrame(transcriptions, columns=["filename", "transcription"])
    temp_df.to_csv(output_csv, mode="a", header=False, index=False, encoding="utf-8")

print(f"Transcriptions saved to {output_csv}")


Using device: cuda
[NeMo I 2025-03-10 19:10:20 mixins:198] _setup_tokenizer: detected an aggregate tokenizer
[NeMo I 2025-03-10 19:10:20 mixins:335] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2025-03-10 19:10:20 mixins:335] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2025-03-10 19:10:20 mixins:335] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2025-03-10 19:10:20 mixins:335] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2025-03-10 19:10:20 mixins:335] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2025-03-10 19:10:20 mixins:335] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2025-03-10 19:10:20 mixins:335] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2025-03-10 19:10:20 mixins:335] Tokenizer SentencePieceTokenizer initialized with 256 tokens
[NeMo I 2025-03-10 19:10:20 mixins:335] Tokenizer SentencePieceTokenizer initialize

[NeMo W 2025-03-10 19:10:27 modelPT:165] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath:
    - /nlsasfs/home/ai4bharat/ai4bharat-pr/speechteam/indicasr_v3/manifests/nemo/vistaar_v3/train/train_bengali.json
    sample_rate: 16000
    batch_size: 8
    shuffle: false
    num_workers: 16
    pin_memory: true
    max_duration: 30.0
    min_duration: 0.2
    is_tarred: false
    tarred_audio_filepaths: null
    shuffle_n: 2048
    bucketing_strategy: synced_randomized
    bucketing_batch_size: null
    is_concat: true
    concat_sampling_technique: temperature
    concat_sampling_temperature: 1.5
    return_language_id: true
    
[NeMo W 2025-03-10 19:10:27 modelPT:172] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method and provide a valid configurati

[NeMo I 2025-03-10 19:10:27 features:289] PADDING: 0
[NeMo I 2025-03-10 19:10:30 rnnt:1667] Vocab size for each language: 256
[NeMo I 2025-03-10 19:10:30 rnnt_models:220] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2025-03-10 19:10:30 rnnt_models:220] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2025-03-10 19:10:31 rnnt_models:220] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2025-03-10 19:10:31 hybrid_rnnt_ctc_bpe_models:105] Creating masks for multi-softmax layer.
[NeMo I 2025-03-10 19:10:31 rnnt_models:220] Using RNNT Loss : warprnnt_numba
    Loss warprnnt_numba_kwargs: {'fastemit_lambda': 0.0, 'clamp': -1.0}
[NeMo I 2025-03-10 19:10:32 save_restore_connector:263] Model EncDecHybridRNNTCTCBPEModel was successfully restored from /root/.cache/huggingface/hub/models--ai4bharat--indic

Transcribing: 100%|██████████| 1/1 [00:00<00:00,  1.14it/s]


Transcribing (2/395): zMJ--i0eCmA.bn_11.wav


...

Transcribing (394/395): December-2023_Bengali_222.wav


Transcribing: 100%|██████████| 1/1 [00:00<00:00, 10.70it/s]


Transcribing (395/395): RTjaB6INtnQ.bn_77.wav


Transcribing: 100%|██████████| 1/1 [00:00<00:00,  1.72it/s]

Transcriptions saved to bengali_transcriptions_dev11.csv


In [10]:
pip install jiwer

Note: you may need to restart the kernel to use updated packages.


In [11]:
import pandas as pd
import sacrebleu
import jiwer  # Import WER library

# Load CSV files
df1 = pd.read_csv("/kaggle/input/en-bn-bn-en-mapping-csv/bn-en dev.csv")
df2 = pd.read_csv("/kaggle/working/NeMo/bengali_transcriptions_dev11.csv")

# Rename columns for consistency
df1.rename(columns={"Audio": "audio_file"}, inplace=True)
df2.rename(columns={"filename": "audio_file", "transcription": "asr_output"}, inplace=True)

# Step 1: Concatenate segments of the same file
df1.sort_values(by=["audio_file", "Start segment"], inplace=True)

concatenated = {}
for _, row in df1.iterrows():
    audio_path = row["audio_file"]
    if audio_path not in concatenated:
        concatenated[audio_path] = {
            "reference_text": row["Source text"],
            "target_translation": row["Translated text"],
            "end_segment": row["End segment"],
        }
    else:
        if row["Start segment"] <= concatenated[audio_path]["end_segment"]:
            concatenated[audio_path]["reference_text"] += " " + row["Source text"]
            concatenated[audio_path]["target_translation"] += " " + row["Translated text"]
            concatenated[audio_path]["end_segment"] = row["End segment"]
        else:
            concatenated[audio_path] = {
                "reference_text": row["Source text"],
                "target_translation": row["Translated text"],
                "end_segment": row["End segment"],
            }

# Convert concatenated dictionary to DataFrame
df1_processed = pd.DataFrame.from_dict(concatenated, orient="index").reset_index()
df1_processed.rename(columns={"index": "audio_file"}, inplace=True)

# Step 2: Merge with ASR transcriptions using "audio_file"
merged = df1_processed.merge(df2, on="audio_file", how="inner")

# Step 3: Compute SacreBLEU scores (sentence-level)
def compute_sacrebleu(reference, candidate):
    reference = reference.lower().strip()
    candidate = candidate.lower().strip()
    return sacrebleu.sentence_bleu(candidate, [reference]).score  # Fix: Pass reference as a list of strings

merged["SacreBLEU Score"] = merged.apply(lambda row: compute_sacrebleu(row["reference_text"], row["asr_output"]), axis=1)

# Compute overall SacreBLEU score (corpus-level)
references = merged["reference_text"].tolist()
candidates = merged["asr_output"].tolist()
overall_bleu = sacrebleu.corpus_bleu(candidates, [references]).score

# Step 4: Compute WER (sentence-level)
def compute_wer(reference, candidate):
    reference = reference.lower().strip()
    candidate = candidate.lower().strip()
    return jiwer.wer(reference, candidate)

merged["WER"] = merged.apply(lambda row: compute_wer(row["reference_text"], row["asr_output"]), axis=1)

# Compute overall WER (corpus-level)
overall_wer = jiwer.wer(references, candidates)

# Print results
print(merged[["audio_file", "SacreBLEU Score", "WER"]])
print("\nOverall SacreBLEU Score:", overall_bleu)
print("Overall WER:", overall_wer)

# Save to CSV
merged.to_csv("sacrebleu_wer_scores.csv", index=False)


                 audio_file  SacreBLEU Score       WER
0     -Mpqqv-wIqI.bn_24.wav        55.666448  0.285714
1     -V0p0CSz5ek.bn_71.wav        33.887144  0.333333
2    -wTFNGnQG-k.bn_161.wav        10.682175  1.000000
3    01fjtFPWenI.bn_100.wav        16.015448  0.526316
4    0CNoNmMdBuE.bn_162.wav        46.396295  0.444444
..                      ...              ...       ...
390   zMJ--i0eCmA.bn_11.wav        43.155330  0.518519
391   zXxifrNp-zg.bn_77.wav        44.632361  0.333333
392  zf1WkHq0k2o.bn_270.wav        17.915646  0.416667
393  zqnkZgDWhVs.bn_124.wav        61.322974  0.222222
394  zx_oMonGdK4.bn_439.wav        57.067458  0.250000

[395 rows x 3 columns]

Overall SacreBLEU Score: 48.02489811374375
Overall WER: 0.39955156950672643


In [12]:
# Create new DataFrame with required columns
final_df = merged[["audio_file", "reference_text", "asr_output", "SacreBLEU Score"]]

# Save to CSV
final_df.to_csv("final_sacrebleu_scores.csv", index=False)

print("CSV file 'final_sacrebleu_scores.csv' created successfully!")


CSV file 'final_sacrebleu_scores.csv' created successfully!


In [13]:
# Print cases where SacreBLEU score is 0
zero_bleu_cases = merged[merged["SacreBLEU Score"] == 0]
print(len(zero_bleu_cases))
if not zero_bleu_cases.empty:
    print("\nCases where SacreBLEU score is 0:")
    for _, row in zero_bleu_cases.iterrows():
        print(f"\nAudio File: {row['audio_file']}")
        print(f"Source Text: {row['reference_text']}")
        print(f"ASR Output: {row['asr_output']}")


60

Cases where SacreBLEU score is 0:

Audio File: 0oqhN5tvLgA.bn_158.wav
Source Text: যদি এইটি গেমের ট্রির মান হয় তাহলে এই আলফা বিটা একটি উইন্ডো
ASR Output: ['']

Audio File: 0oqhN5tvLgA.bn_275.wav
Source Text: এই বিটা বলছে আমি অন্তত 80
ASR Output: ['']

Audio File: 0oqhN5tvLgA.bn_283.wav
Source Text: বাম এবং ডান মানে কি
ASR Output: ['']

Audio File: 13_1_Bengali_27.wav
Source Text: মিশ্ম দূমা মসা হদদ তেমা
ASR Output: [' মিষ্ম দুমা মশা হ্রদ তেমা']

Audio File: 5kfjNXKnKIQ.bn_22.wav
Source Text: কিন্তু এবার E আলাদা হবে E ভেক্টর কে লিখি সেটি হবে Q বাই 4 পাই এপসাইলন 0 K r স্কয়ার r এর দিকে
ASR Output: ['']

Audio File: 6kRYg1QVcWo.bn_272.wav
Source Text: সুতরাং আমি এখানে থামব
ASR Output: ['']

Audio File: 9Nc6spNjDvI.bn_36.wav
Source Text: পরিবেশ কী
ASR Output: ['']

Audio File: April-2016_Bengali_29.wav
Source Text: কখনো কখনো মনে হয় যে এখন তো ভারত সরকার রেল পরিষেবার মাধ্যমে লাতুরে জল পৌঁছে দেয় তা বিশ্বের কাছে এক বড় খবর হয়ে ওঠে
ASR Output: ['']

Audio File: April-2018_Bengali_126.wa